In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from torch.profiler import profile, record_function, ProfilerActivity

# Check what you are working with
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
    device = "cuda"
else:
    print("Running on CPU")
    device = "cpu"

print("Device:", device)

PyTorch version: 2.11.0+cu128
GPU available: True
GPU name: Tesla T4
GPU memory: 15.637086208 GB
Device: cuda


In [2]:
# ================================================================
# UNIVERSAL PROFILING TEMPLATE
# Step 1: paste the model here
# Step 2: run this cell
# Step 3: read the output
# ================================================================

# --- PASTE INTERVIEWER'S MODEL HERE ---
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(1024, 4096)
        self.act    = nn.ReLU()
        self.layer2 = nn.Linear(4096, 1024)
        self.layer3 = nn.Linear(1024, 512)
        self.layer4 = nn.Linear(512, 256)

    def forward(self, x):
        x = self.act(self.layer1(x))
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x
# --- END OF MODEL ---


# --- SETUP ---
model = FeedForward().to(device).eval()
x = torch.randn(32, 1024).to(device)   # batch=32, input size matches model


# --- WARMUP (always do this before measuring) ---
print("Warming up...")
with torch.no_grad():
    for _ in range(10):
        _ = model(x)

# If GPU — wait for all GPU operations to finish before timing
if device == "cuda":
    torch.cuda.synchronize()

print("Warmup done.")


# --- BASELINE MEASUREMENT ---
print("\nMeasuring baseline...")
times = []

with torch.no_grad():
    for _ in range(100):
        if device == "cuda":
            torch.cuda.synchronize()    # wait for GPU to be ready

        start = time.perf_counter()
        out = model(x)

        if device == "cuda":
            torch.cuda.synchronize()    # wait for GPU to finish

        end = time.perf_counter()
        times.append((end - start) * 1000)

baseline_ms = sum(times) / len(times)
print(f"Baseline latency: {baseline_ms:.3f} ms")
print(f"Output shape: {out.shape}")


# --- MEMORY ---
total_params = sum(p.numel() for p in model.parameters())
memory_fp32 = total_params * 4 / 1e9
print(f"\nTotal parameters: {total_params:,}")
print(f"Model memory (FP32): {memory_fp32:.4f} GB")

if device == "cuda":
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.4f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.4f} GB")

Warming up...
Warmup done.

Measuring baseline...
Baseline latency: 0.471 ms
Output shape: torch.Size([32, 256])

Total parameters: 9,049,856
Model memory (FP32): 0.0362 GB
GPU memory used: 0.0459 GB
GPU memory reserved: 0.0587 GB


In [3]:
# ================================================================
# PROFILING CELL
# This tells you WHICH operations take the most time
# ================================================================

# Choose activities based on device
activities = [ProfilerActivity.CPU]
if device == "cuda":
    activities.append(ProfilerActivity.CUDA)

print("Running profiler...")

with profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
    with_stack=False
) as prof:
    with record_function("model_forward"):
        with torch.no_grad():
            out = model(x)
        if device == "cuda":
            torch.cuda.synchronize()

# --- PRINT RESULTS ---
print("\n" + "="*80)
print("TOP 10 OPERATIONS BY TIME")
print("="*80)

# On GPU sort by CUDA time, on CPU sort by CPU time
sort_key = "cuda_time_total" if device == "cuda" else "cpu_time_total"

print(prof.key_averages().table(
    sort_by=sort_key,
    row_limit=10
))

# --- WHAT TO LOOK FOR ---
print("\n" + "="*80)
print("HOW TO READ THIS OUTPUT:")
print("="*80)
print("Name        → what operation ran (aten::linear = matrix multiply)")
print("CPU total   → how much CPU time it took")
print("CUDA total  → how much GPU time it took (if GPU available)")
print("# Calls     → how many times this ran")
print("Memory      → how much memory it allocated")
print("\nThe TOP ROW = YOUR BOTTLENECK. That is what you optimize first.")

Running profiler...

TOP 10 OPERATIONS BY TIME
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          model_forward         0.00%       0.000us         0.00%       0.000us       0.000us     802.074us       195.85%     802.074us     802.074us        

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [4]:
# ================================================================
# OPTIMIZATION CELL
# Run this after profiling to see improvement
# ================================================================

results = {}
results["baseline_fp32"] = baseline_ms
print(f"Baseline (FP32): {baseline_ms:.3f} ms")
print("-" * 40)


# --- OPTIMIZATION 1: FP16 ---
print("\nOptimization 1: FP16")
model_fp16 = FeedForward().half().to(device).eval()
x_fp16 = x.half()

with torch.no_grad():
    for _ in range(10): _ = model_fp16(x_fp16)   # warmup
    if device == "cuda": torch.cuda.synchronize()

times_fp16 = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        start = time.perf_counter()
        out = model_fp16(x_fp16)
        if device == "cuda": torch.cuda.synchronize()
        times_fp16.append((time.perf_counter() - start) * 1000)

fp16_ms = sum(times_fp16) / len(times_fp16)
results["fp16"] = fp16_ms
print(f"FP16 latency: {fp16_ms:.3f} ms  →  {baseline_ms/fp16_ms:.2f}x faster")


# --- OPTIMIZATION 2: torch.compile ---
print("\nOptimization 2: torch.compile")
print("(First run is slow — compiling... wait for it)")
model_compiled = torch.compile(FeedForward().to(device).eval())

with torch.no_grad():
    for _ in range(15): _ = model_compiled(x)    # more warmup for compile
    if device == "cuda": torch.cuda.synchronize()

times_compiled = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        start = time.perf_counter()
        out = model_compiled(x)
        if device == "cuda": torch.cuda.synchronize()
        times_compiled.append((time.perf_counter() - start) * 1000)

compiled_ms = sum(times_compiled) / len(times_compiled)
results["compiled"] = compiled_ms
print(f"Compiled latency: {compiled_ms:.3f} ms  →  {baseline_ms/compiled_ms:.2f}x faster")


# --- OPTIMIZATION 3: FP16 + compile ---
print("\nOptimization 3: FP16 + torch.compile")
model_best = torch.compile(FeedForward().half().to(device).eval())

with torch.no_grad():
    for _ in range(15): _ = model_best(x_fp16)
    if device == "cuda": torch.cuda.synchronize()

times_best = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        start = time.perf_counter()
        out = model_best(x_fp16)
        if device == "cuda": torch.cuda.synchronize()
        times_best.append((time.perf_counter() - start) * 1000)

best_ms = sum(times_best) / len(times_best)
results["fp16_compiled"] = best_ms
print(f"FP16+Compiled: {best_ms:.3f} ms  →  {baseline_ms/best_ms:.2f}x faster")


# --- FINAL SUMMARY TABLE ---
print("\n" + "="*55)
print("FINAL RESULTS SUMMARY")
print("="*55)
print(f"{'Method':<25} {'Latency':>10} {'Speedup':>10}")
print("-"*55)
for name, ms in results.items():
    speedup = baseline_ms / ms
    print(f"{name:<25} {ms:>8.3f}ms {speedup:>9.2f}x")
print("="*55)

Baseline (FP32): 0.471 ms
----------------------------------------

Optimization 1: FP16
FP16 latency: 1.149 ms  →  0.41x faster

Optimization 2: torch.compile
(First run is slow — compiling... wait for it)


W0527 07:03:36.690000 580 torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


Compiled latency: 0.660 ms  →  0.71x faster

Optimization 3: FP16 + torch.compile
FP16+Compiled: 0.407 ms  →  1.16x faster

FINAL RESULTS SUMMARY
Method                       Latency    Speedup
-------------------------------------------------------
baseline_fp32                0.471ms      1.00x
fp16                         1.149ms      0.41x
compiled                     0.660ms      0.71x
fp16_compiled                0.407ms      1.16x


# model 2


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from torch.profiler import profile, record_function, ProfilerActivity

# ================================================================
# STEP 1 — THE COMPLETE MODEL WITH KV CACHE
# ================================================================

class TransformerBlockWithKVCache(nn.Module):
    def __init__(self, dim=512, num_heads=8):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.ff = nn.Linear(dim, dim * 4)
        self.ff2 = nn.Linear(dim * 4, dim)
        self.k_cache = None
        self.v_cache = None

    def forward(self, x, use_cache=False):
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        if use_cache:
            if self.k_cache is None:
                self.k_cache = k
                self.v_cache = v
            else:
                self.k_cache = torch.cat([self.k_cache, k], dim=1)
                self.v_cache = torch.cat([self.v_cache, v], dim=1)
            k_for_attn = self.k_cache
            v_for_attn = self.v_cache
        else:
            k_for_attn = k
            v_for_attn = v

        scale = self.head_dim ** 0.5
        scores = torch.bmm(q, k_for_attn.transpose(1, 2)) / scale
        weights = torch.softmax(scores, dim=-1)
        attn_out = torch.bmm(weights, v_for_attn)
        out = self.out_proj(attn_out)
        out = self.norm(out + x)
        out = self.ff2(F.relu(self.ff(out)))
        return out

    def clear_cache(self):
        self.k_cache = None
        self.v_cache = None


# ================================================================
# STEP 2 — WITHOUT KV CACHE
# every step feeds the growing sequence from scratch
# ================================================================

model = TransformerBlockWithKVCache().eval()
seq_length = 50
times_no_cache = []

print("=== WITHOUT KV CACHE ===")
with torch.no_grad():
    for step in range(seq_length):
        x = torch.randn(1, step + 1, 512)
        start = time.perf_counter()
        out = model(x, use_cache=False)
        end = time.perf_counter()
        times_no_cache.append((end - start) * 1000)

print(f"Step 1  latency: {times_no_cache[0]:.3f} ms")
print(f"Step 25 latency: {times_no_cache[24]:.3f} ms")
print(f"Step 50 latency: {times_no_cache[49]:.3f} ms")
print(f"Total time:      {sum(times_no_cache):.3f} ms")


# ================================================================
# STEP 3 — WITH KV CACHE
# every step feeds only 1 new token, cache handles the rest
# ================================================================

model.clear_cache()
times_with_cache = []

print("\n=== WITH KV CACHE ===")
with torch.no_grad():
    for step in range(seq_length):
        x = torch.randn(1, 1, 512)
        start = time.perf_counter()
        out = model(x, use_cache=True)
        end = time.perf_counter()
        times_with_cache.append((end - start) * 1000)

print(f"Step 1  latency: {times_with_cache[0]:.3f} ms")
print(f"Step 25 latency: {times_with_cache[24]:.3f} ms")
print(f"Step 50 latency: {times_with_cache[49]:.3f} ms")
print(f"Total time:      {sum(times_with_cache):.3f} ms")


# ================================================================
# STEP 4 — PROFILER ON WITHOUT CACHE (find bottleneck)
# ================================================================

model.clear_cache()
x_profile = torch.randn(1, 10, 512)

print("\n=== PROFILER — WITHOUT CACHE ===")
with profile(
    activities=[ProfilerActivity.CPU],
    record_shapes=True,
    profile_memory=True
) as prof:
    with record_function("no_cache_forward"):
        with torch.no_grad():
            out = model(x_profile, use_cache=False)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=8))


# ================================================================
# STEP 5 — FINAL COMPARISON SUMMARY
# ================================================================

print("\n" + "="*55)
print("FINAL COMPARISON")
print("="*55)
print(f"{'Step':<8} {'No Cache (ms)':>15} {'With Cache (ms)':>16} {'Speedup':>10}")
print("-"*55)
for step in [0, 9, 24, 49]:
    no_c = times_no_cache[step]
    with_c = times_with_cache[step]
    speedup = no_c / with_c
    print(f"{step+1:<8} {no_c:>15.3f} {with_c:>16.3f} {speedup:>9.2f}x")
print("="*55)
print(f"\nTotal without cache: {sum(times_no_cache):.2f} ms")
print(f"Total with cache:    {sum(times_with_cache):.2f} ms")
print(f"Overall speedup:     {sum(times_no_cache)/sum(times_with_cache):.2f}x")

=== WITHOUT KV CACHE ===
Step 1  latency: 56.502 ms
Step 25 latency: 3.863 ms
Step 50 latency: 7.612 ms
Total time:      267.621 ms

=== WITH KV CACHE ===
Step 1  latency: 1.582 ms
Step 25 latency: 1.225 ms
Step 50 latency: 1.027 ms
Total time:      64.905 ms

=== PROFILER — WITHOUT CACHE ===
---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
           no_cache_forward        16.02%     704.313us       100.00%       4.395ms       4.395ms      20.00 KB    -301.17 KB             1  
               aten::linear         2.23%      97.953us        63.29%       2.782ms     463.612us     180.00 KB           0 B             

# model 3


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from torch.profiler import profile, record_function, ProfilerActivity

# ================================================================
# MODEL 3 — MINI LLM
# ================================================================

class MiniLLM(nn.Module):
    def __init__(self, vocab_size=32000, dim=512, num_layers=6, num_heads=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=dim,
                nhead=num_heads,
                dim_feedforward=dim * 4,
                batch_first=True
            )
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(dim)
        self.output_proj = nn.Linear(dim, vocab_size)

    def forward(self, token_ids):
        x = self.embedding(token_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.output_proj(x)


# ================================================================
# SETUP
# ================================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = MiniLLM().to(device).eval()

batch_size = 8
seq_len = 128
token_ids = torch.randint(0, 32000, (batch_size, seq_len)).to(device)

total_params = sum(p.numel() for p in model.parameters())
memory_fp32_gb = total_params * 4 / 1e9
print(f"Total parameters: {total_params:,}")
print(f"Model memory FP32: {memory_fp32_gb:.3f} GB")


# ================================================================
# STEP 1 — WARMUP + BASELINE
# ================================================================

print("\n--- WARMUP ---")
with torch.no_grad():
    for _ in range(5):
        _ = model(token_ids)
if device == "cuda":
    torch.cuda.synchronize()
print("Warmup done.")

print("\n--- BASELINE MEASUREMENT ---")
times_baseline = []
with torch.no_grad():
    for _ in range(50):
        if device == "cuda": torch.cuda.synchronize()
        start = time.perf_counter()
        out = model(token_ids)
        if device == "cuda": torch.cuda.synchronize()
        times_baseline.append((time.perf_counter() - start) * 1000)

baseline_ms = sum(times_baseline) / len(times_baseline)
print(f"Baseline latency (FP32): {baseline_ms:.3f} ms")
print(f"Output shape: {out.shape}")


# ================================================================
# STEP 2 — PROFILER
# ================================================================

print("\n--- PROFILER OUTPUT ---")
activities = [ProfilerActivity.CPU]
if device == "cuda":
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True
) as prof:
    with record_function("minillm_forward"):
        with torch.no_grad():
            out = model(token_ids)
        if device == "cuda":
            torch.cuda.synchronize()

sort_key = "cuda_time_total" if device == "cuda" else "cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=10))


# ================================================================
# STEP 3 — OPTIMIZATION 1 — FP16
# ================================================================

print("\n--- OPTIMIZATION 1: FP16 ---")
model_fp16 = MiniLLM().half().to(device).eval()

with torch.no_grad():
    for _ in range(5): _ = model_fp16(token_ids)
if device == "cuda": torch.cuda.synchronize()

times_fp16 = []
with torch.no_grad():
    for _ in range(50):
        if device == "cuda": torch.cuda.synchronize()
        start = time.perf_counter()
        out = model_fp16(token_ids)
        if device == "cuda": torch.cuda.synchronize()
        times_fp16.append((time.perf_counter() - start) * 1000)

fp16_ms = sum(times_fp16) / len(times_fp16)
fp16_memory_gb = total_params * 2 / 1e9
print(f"FP16 latency:  {fp16_ms:.3f} ms")
print(f"FP16 memory:   {fp16_memory_gb:.3f} GB")
print(f"Speedup:       {baseline_ms/fp16_ms:.2f}x")
print(f"Memory saving: {memory_fp32_gb/fp16_memory_gb:.1f}x smaller")


# ================================================================
# STEP 4 — OPTIMIZATION 2 — torch.compile
# ================================================================

print("\n--- OPTIMIZATION 2: torch.compile ---")
print("Compiling... first run will be slow, wait for it...")
model_compiled = torch.compile(MiniLLM().to(device).eval())

with torch.no_grad():
    for _ in range(15): _ = model_compiled(token_ids)
if device == "cuda": torch.cuda.synchronize()

times_compiled = []
with torch.no_grad():
    for _ in range(50):
        if device == "cuda": torch.cuda.synchronize()
        start = time.perf_counter()
        out = model_compiled(token_ids)
        if device == "cuda": torch.cuda.synchronize()
        times_compiled.append((time.perf_counter() - start) * 1000)

compiled_ms = sum(times_compiled) / len(times_compiled)
print(f"Compiled latency: {compiled_ms:.3f} ms")
print(f"Speedup:          {baseline_ms/compiled_ms:.2f}x")


# ================================================================
# STEP 5 — OPTIMIZATION 3 — FP16 + compile
# ================================================================

print("\n--- OPTIMIZATION 3: FP16 + torch.compile ---")
model_best = torch.compile(MiniLLM().half().to(device).eval())

with torch.no_grad():
    for _ in range(15): _ = model_best(token_ids)
if device == "cuda": torch.cuda.synchronize()

times_best = []
with torch.no_grad():
    for _ in range(50):
        if device == "cuda": torch.cuda.synchronize()
        start = time.perf_counter()
        out = model_best(token_ids)
        if device == "cuda": torch.cuda.synchronize()
        times_best.append((time.perf_counter() - start) * 1000)

best_ms = sum(times_best) / len(times_best)
print(f"FP16 + Compiled latency: {best_ms:.3f} ms")
print(f"Speedup:                 {baseline_ms/best_ms:.2f}x")


# ================================================================
# STEP 6 — MEMORY CHECK ON GPU
# ================================================================

if device == "cuda":
    print("\n--- GPU MEMORY USAGE ---")
    print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.3f} GB")
    print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.3f} GB")


# ================================================================
# STEP 7 — FINAL SUMMARY
# ================================================================

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY — MiniLLM")
print("="*60)
print(f"{'Method':<25} {'Latency':>10} {'Speedup':>10} {'Memory':>10}")
print("-"*60)
print(f"{'Baseline FP32':<25} {baseline_ms:>9.3f}ms {'1.00x':>10} {memory_fp32_gb:>8.3f}GB")
print(f"{'FP16':<25} {fp16_ms:>9.3f}ms {baseline_ms/fp16_ms:>9.2f}x {fp16_memory_gb:>8.3f}GB")
print(f"{'torch.compile':<25} {compiled_ms:>9.3f}ms {baseline_ms/compiled_ms:>9.2f}x {memory_fp32_gb:>8.3f}GB")
print(f"{'FP16 + compile':<25} {best_ms:>9.3f}ms {baseline_ms/best_ms:>9.2f}x {fp16_memory_gb:>8.3f}GB")
print("="*60)
print(f"\nBest optimization: FP16 + compile")
print(f"Latency: {baseline_ms:.3f}ms → {best_ms:.3f}ms")
print(f"Memory:  {memory_fp32_gb:.3f}GB → {fp16_memory_gb:.3f}GB")
print(f"Speedup: {baseline_ms/best_ms:.2f}x faster, {memory_fp32_gb/fp16_memory_gb:.1f}x less memory")

Using device: cuda
Total parameters: 51,715,328
Model memory FP32: 0.207 GB

--- WARMUP ---
Warmup done.

--- BASELINE MEASUREMENT ---
Baseline latency (FP32): 20.031 ms
Output shape: torch.Size([8, 128, 32000])

--- PROFILER OUTPUT ---
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  